# Basic usage

List the checks, then run them on the sample data.

Import the reusable row-based API from `off_data_quality`.

`from off_data_quality import checks` gives us the public surface for listing
shipped checks and running them on loaded rows.

In [1]:
from off_data_quality import checks

Import DuckDB as a convenient loader for the bundled sample data.

DuckDB is not required by `off_data_quality`: you can load rows with another
library and still pass the resulting rows to `checks.run(...)` as long as the
input matches one supported input shape.

In [2]:
import duckdb

In [3]:
import json
import os

DATA_DIR = "examples/data" if os.path.isdir("examples/data") else "../data"
PARQUET_SAMPLE = f"{DATA_DIR}/products.parquet"
SELECTED_CHECK_IDS = (
    "en:quantity-to-be-completed",
    "en:nutrition-data-per-serving-serving-quantity-is-not-recognized",
    "en:serving-quantity-over-product-quantity",
)

Load the bundled OFF Parquet sample directly.

In [4]:
rows = duckdb.read_parquet(PARQUET_SAMPLE)

Look at the checks in the library.

In [5]:
all_checks = checks.list()
print("All checks")
print(json.dumps([check.id for check in all_checks], indent=2))

All checks
[
  "ca:source-of-fibre-claim-but-fibre-below-threshold",
  "ca:source-of-omega-3-polyunsaturates-claim-but-omega-3-below-threshold",
  "ca:trans-fat-free-claim-but-nutrition-does-not-meet-conditions",
  "en:alcoholic-and-non-alcoholic-categories",
  "en:code-empty",
  "en:code-missing",
  "en:created-missing",
  "en:created-zero",
  "en:incompatible-categories-plant-milk-and-dairy",
  "en:no-fat-label-claim-but-fat-above-0.5",
  "en:nutrition-data-per-serving-serving-quantity-is-not-recognized",
  "en:product-name-to-be-completed",
  "en:product-quantity-over-10kg",
  "en:product-quantity-over-30kg",
  "en:product-quantity-under-1g",
  "en:quantity-not-recognized",
  "en:quantity-to-be-completed",
  "en:saturated-fat-free-label-claim-but-fat-above-0.1",
  "en:serving-quantity-defined-but-quantity-undefined",
  "en:serving-quantity-less-than-product-quantity-divided-by-1000",
  "en:serving-quantity-over-500g",
  "en:serving-quantity-over-product-quantity",
  "en:serving-quan

Run all checks on the loaded rows.

In [6]:
all_findings = checks.run(rows)
print()
print("All findings")
print(json.dumps({"count": len(all_findings)}, indent=2))


All findings
{
  "count": 1003
}


Run a smaller set of checks.

In [7]:
selected_checks = checks.list(check_ids=SELECTED_CHECK_IDS)
selected_findings = checks.run(rows, check_ids=SELECTED_CHECK_IDS)
selected_example = None
for finding in selected_findings:
    selected_example = {
        "product_id": finding.product_id,
        "check_id": finding.check_id,
        "severity": finding.severity,
    }
    break

print()
print("Selected checks")
print(json.dumps([check.id for check in selected_checks], indent=2))

print()
print("Selected findings")
print(
    json.dumps(
        {
            "count": len(selected_findings),
            "example": selected_example,
        },
        indent=2,
    )
)


Selected checks
[
  "en:nutrition-data-per-serving-serving-quantity-is-not-recognized",
  "en:quantity-to-be-completed",
  "en:serving-quantity-over-product-quantity"
]

Selected findings
{
  "count": 552,
  "example": {
    "product_id": "0000417012009",
    "check_id": "en:nutrition-data-per-serving-serving-quantity-is-not-recognized",
    "severity": "warning"
  }
}
